# 03 — Prompt variants (single-variable vs the shipped baseline)

Can a prompt tweak beat the already-strong baseline prompt (which already carries the full FO taxonomy)? 6 arms, **select on `val_id`**, **confirm the winner on held-out `val_ood`**, ship only if the OOD bootstrap CI does not overlap the baseline. See `README.md` + `context/03-prompt-variants/CONTEXT.md`. Engine + drivers hold the logic; these cells generate the run.

In [ ]:
# bootstrap
import sys, logging
from pathlib import Path
EXP_DIR = Path.cwd()
REPO = EXP_DIR
while REPO != REPO.parent and not ((REPO / '.git').exists() or (REPO / 'src').is_dir()):
    REPO = REPO.parent
for p in (EXP_DIR / '_models', REPO / 'src'):
    if p.is_dir():
        sys.path.insert(0, str(p))
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s %(message)s', datefmt='%H:%M:%S')
import prompt_variants as pv
ARMS = ['a0_noFO', 'a1_v0', 'a2_decisive', 'a3_instrument', 'a4_negexem', 'a5_baredigit']
RUNS = EXP_DIR / 'runs'; RUNS.mkdir(parents=True, exist_ok=True)
print('arms:', ARMS)

## Stage 1 — select on `val_id` (2252 Q, cholecystectomy)
Winner must show a broad lift (fo_class AND number up), not a chole-specific leaf.

In [ ]:
sel_rows = pv.run_arms(ARMS, 'val_id', RUNS)
pv.write_rows(sel_rows, RUNS / 'select_results.csv')
import pandas as pd
sel = pd.DataFrame(sel_rows).sort_values('acc', ascending=False)
print(sel[['arm', 'acc', 'ci_low', 'ci_high', 'fo_class', 'number']].to_string(index=False))

## Stage 2 — confirm on held-out `val_ood` (4000 Q, Sigmoid) + ship rule
`_tools/run_confirm.py` picks the winning challenger, re-evaluates `[winner, a1_v0]` on `val_ood`, and ships only if the OOD CIs do not overlap (else faithful negative).

In [ ]:
# runs the pre-registered confirmation + verdict (also reports a0_noFO = FO-grounding delta)
import runpy
runpy.run_path(str(EXP_DIR / '_tools' / 'run_confirm.py'), run_name='__main__')

## Result
Fill `RESULTS.csv` + the README ladder from `runs/confirm_results.csv`. Faithful negative is a valid outcome (Constitution §VIII.6).